# Titanic: Machine Learning from Disaster
## Comprehensive Model Comparison and Survival Prediction Pipeline

**Author:** CINTEL Task Submission  
**Dataset:** [Kaggle Titanic Competition](https://www.kaggle.com/competitions/titanic/data)  
**Target Variable:** `Survived` (0 = No, 1 = Yes)  

---
### Project Objective
The objective of this project is to build an end-to-end Machine Learning classification workflow that accurately predicts passenger survival on the RMS Titanic. We rigorously compare multiple classification algorithms (**Logistic Regression**, **Random Forest**, **Gradient Boosting**, and **XGBoost**) using standard classification metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC, and 5-Fold Cross-Validation).

Strict methodologies are implemented to avoid **data leakage** through the use of Scikit-Learn `Pipeline` and `ColumnTransformer`.

## 1. Environment Setup and Data Loading
We import all standard scientific computing, visualization, and machine learning libraries.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb

# Set aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print('Libraries successfully imported!')

In [ ]:
# Load datasets
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

print(f'Training set shape:   {train_df.shape}')
print(f'Test set (Kaggle) shape: {test_df.shape}')
train_df.head()

### Dataset Inspection & Summary Statistics
Inspecting column types, missing values, duplicates, and statistical distribution.

In [ ]:
# Summary Info & Missing Values
print('--- Dataset Info ---')
print(train_df.info())
print('\n--- Missing Values Count ---')
missing_vals = train_df.isnull().sum()
print(missing_vals[missing_vals > 0])
print(f'\nNumber of duplicate rows: {train_df.duplicated().sum()}')

# Descriptive Statistics
train_df.describe().T

## 2. Feature Rationale & Selection
- **`PassengerId`**: Arbitrary unique identifier; carries no predictive signal and must be dropped to prevent overfitting.
- **`Name`**: High-cardinality string; not directly usable, but contains valuable social honorifics (`Mr`, `Mrs`, `Miss`, `Master`, etc.) that proxy social class, marital status, and age.
- **`Pclass`**: Ticket class (1st, 2nd, 3rd); direct proxy for socio-economic status and cabin deck position.
- **`Sex`**: Crucial demographic feature reflecting the 'women and children first' evacuation protocol.
- **`Age`**: Continuous demographic feature; young children had higher survival priority.
- **`SibSp` & `Parch`**: Family counts on board; useful when combined into aggregate travel party dynamics.
- **`Ticket`**: High cardinality string with inconsistent formatting; dropped to avoid noise.
- **`Fare`**: Continuous metric indicating financial tier and location on the ship.
- **`Cabin`**: Over 77% missing values; raw cabin numbers are unreliable, though deck letters can indicate position.
- **`Embarked`**: Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton).

## 3. Feature Engineering
We construct three high-impact domain features:
1. **`Title`**: Extracted from `Name` using regex (`Mr`, `Mrs`, `Miss`, `Master`, `Rare`).
2. **`FamilySize`**: `SibSp + Parch + 1` (total party size including the passenger).
3. **`IsAlone`**: Binary indicator (`1` if `FamilySize == 1`, else `0`). Solo travelers faced distinct survival dynamics compared to coordinated family units.

In [ ]:
def extract_title(name: str) -> str:
    title_search = re.search(r' ([A-Za-z]+)\.', name)
    if not title_search:
        return 'Rare'
    title = title_search.group(1)
    if title in ['Mlle', 'Ms']:
        return 'Miss'
    elif title in ['Mme']:
        return 'Mrs'
    elif title in ['Mr', 'Miss', 'Mrs', 'Master']:
        return title
    else:
        return 'Rare'

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['Title'] = df['Name'].apply(extract_title)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    return df

train_fe = engineer_features(train_df)
test_fe = engineer_features(test_df)
train_fe[['Name', 'Title', 'SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

## 4. Exploratory Data Analysis (EDA)
Visualizing survival patterns across demographic and engineered attributes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# 1. Overall Survival
sns.countplot(data=train_fe, x='Survived', ax=axes[0, 0], palette='coolwarm')
axes[0, 0].set_title('1. Overall Survival Distribution (0=Died, 1=Survived)')
axes[0, 0].set_xticklabels(['Died (61.6%)', 'Survived (38.4%)'])

# 2. Survival by Sex
sns.barplot(data=train_fe, x='Sex', y='Survived', ax=axes[0, 1], palette='Blues_d', errorbar=None)
axes[0, 1].set_title('2. Survival Rate by Sex')
axes[0, 1].set_ylabel('Survival Rate')

# 3. Survival by Pclass
sns.barplot(data=train_fe, x='Pclass', y='Survived', ax=axes[0, 2], palette='magma', errorbar=None)
axes[0, 2].set_title('3. Survival Rate by Pclass')
axes[0, 2].set_ylabel('Survival Rate')

# 4. Age Distribution by Survival
sns.kdeplot(data=train_fe, x='Age', hue='Survived', common_norm=False, fill=True, ax=axes[1, 0], palette='Set1')
axes[1, 0].set_title('4. Age Distribution by Survival')

# 5. Fare Distribution
sns.boxplot(data=train_fe, x='Survived', y='Fare', ax=axes[1, 1], palette='pastel')
axes[1, 1].set_ylim(0, 250)
axes[1, 1].set_title('5. Fare Distribution by Survival (Capped at 250)')

# 6. Survival by FamilySize
sns.barplot(data=train_fe, x='FamilySize', y='Survived', ax=axes[1, 2], palette='viridis', errorbar=None)
axes[1, 2].set_title('6. Survival Rate by Family Size')

plt.tight_layout()
plt.show()

### EDA Observations & Patterns
1. **Overall Survival:** Approximately 38.4% of passengers survived, creating a baseline class distribution without extreme imbalance.
2. **Survival by Sex:** Females had a ~74% survival rate compared to ~19% for males, verifying the maritime 'women and children first' protocol.
3. **Survival by Pclass:** First-class passengers survived at >60%, whereas 3rd-class passengers had <25% survival, highlighting socioeconomic privileges in lifeboat access.
4. **Age Distribution:** Children under 10 exhibited a noticeable survival bump, while adults aged 20–35 experienced the highest casualty density.
5. **Fare Distribution:** Survivors paid significantly higher median fares, correlating with upper deck accommodations.
6. **Family Size Dynamics:** Small families (size 2–4) experienced survival rates >55%, while solo travelers (~30%) and large families (size >= 5) suffered low survival due to coordination friction during evacuation.

In [ ]:
plt.figure(figsize=(9, 6))
numeric_cols = train_fe[['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']].dropna()
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features', fontsize=13, weight='bold')
plt.show()

## 5. Preprocessing Pipeline & Data Splitting (No Data Leakage)
To strictly prevent data leakage:
- All imputations and feature scaling are learned **solely from the training split** via Scikit-Learn `Pipeline` and `ColumnTransformer`.
- Numerical features: Median Imputation + `StandardScaler`.
- Categorical features: Most-Frequent Imputation + `OneHotEncoder(handle_unknown='ignore')`.
- We split data into **80% Training** and **20% Validation** using stratified sampling (`stratify=y`, `random_state=42`).

In [ ]:
feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Title', 'FamilySize', 'IsAlone']
X = train_fe[feature_cols]
y = train_fe['Survived']
X_test_kaggle = test_fe[feature_cols]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']
categorical_features = ['Sex', 'Embarked', 'Pclass', 'Title']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)
print('ColumnTransformer preprocessor initialized.')

## 6. Model Training & 5-Fold Stratified Cross-Validation
We benchmark four distinct model families:
1. **Logistic Regression**: Linear baseline with L2 regularization.
2. **Random Forest Classifier**: Bagged ensemble of decision trees.
3. **Gradient Boosting Classifier**: Sequential boosting minimizing deviance loss.
4. **XGBoost Classifier**: Optimized gradient boosting with extreme tree pruning.

In [ ]:
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', xgb.XGBClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42, eval_metric='logloss'))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores.mean()
    print(f'{name:<22} 5-Fold CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})')

## 7. Model Evaluation & Comparison
Evaluating trained models on the held-out validation set (20%).

In [ ]:
metrics_list = []
y_preds = {}
y_probs = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    probs = model.predict_proba(X_val)[:, 1]
    y_preds[name] = preds
    y_probs[name] = probs
    
    metrics_list.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_val, preds), 4),
        'Precision': round(precision_score(y_val, preds), 4),
        'Recall': round(recall_score(y_val, preds), 4),
        'F1 Score': round(f1_score(y_val, preds), 4),
        'ROC-AUC': round(roc_auc_score(y_val, probs), 4),
        'CV Accuracy (Mean)': round(cv_results[name], 4)
    })

comparison_df = pd.DataFrame(metrics_list)
comparison_df

## 8. Confusion Matrix Visualizations
Explicitly labeling True Negatives (TN), False Positives (FP), False Negatives (FN), and True Positives (TP).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(y_preds.items()):
    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()
    labels = np.array([[f'TN\n{tn}', f'FP\n{fp}'], [f'FN\n{fn}', f'TP\n{tp}']])
    
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False, ax=axes[idx],
                annot_kws={'size': 13, 'weight': 'bold'},
                xticklabels=['Pred 0 (Died)', 'Pred 1 (Survived)'],
                yticklabels=['Actual 0 (Died)', 'Actual 1 (Survived)'])
    axes[idx].set_title(f'{name}\nAccuracy: {accuracy_score(y_val, y_pred):.4f} | F1: {f1_score(y_val, y_pred):.4f}')

plt.tight_layout()
plt.show()

## 9. Receiver Operating Characteristic (ROC) Curves

In [ ]:
plt.figure(figsize=(9, 7))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for (name, y_prob), col in zip(y_probs.items(), colors):
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc_val = roc_auc_score(y_val, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.4f})', color=col, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.title('ROC Curves Comparison on Held-Out Validation Set', fontsize=13, weight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.show()

## 10. Hyperparameter Tuning (Random Forest via GridSearchCV)
Optimizing tree depth, estimator counts, and split criteria with 5-fold cross validation.

In [ ]:
param_grid = {
    'classifier__n_estimators': [100],
    'classifier__max_depth': [4, 6],
    'classifier__min_samples_split': [2, 5]
}
rf_base = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring='accuracy', n_jobs=1, verbose=1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best 5-Fold CV Accuracy: {grid_search.best_score_:.4f}')

best_pred = best_rf.predict(X_val)
print(f'Validation Accuracy: {accuracy_score(y_val, best_pred):.4f}')
print(f'Validation F1-Score: {f1_score(y_val, best_pred):.4f}')

## 11. Feature Importance Analysis
Extracting feature importances from the tuned Random Forest model.

In [ ]:
fitted_prep = best_rf.named_steps['preprocessor']
ohe_cols = fitted_prep.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
all_feat_names = list(numeric_features) + list(ohe_cols)

importances = best_rf.named_steps['classifier'].feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': all_feat_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=feat_imp_df.head(15), x='Importance', y='Feature', hue='Feature', palette='viridis', legend=False)
plt.title('Top 15 Most Influential Features (Random Forest)', fontsize=13, weight='bold')
plt.xlabel('Gini Importance')
plt.show()

## 12. Sample Passenger Inference Interface
Testing the trained model on hypothetical passenger scenarios.

In [ ]:
sample_passengers = pd.DataFrame([
    {
        'Pclass': 1, 'Sex': 'female', 'Age': 28.0, 'SibSp': 1, 'Parch': 0,
        'Fare': 80.0, 'Embarked': 'S', 'Title': 'Mrs', 'FamilySize': 2, 'IsAlone': 0
    },
    {
        'Pclass': 3, 'Sex': 'male', 'Age': 22.0, 'SibSp': 0, 'Parch': 0,
        'Fare': 7.25, 'Embarked': 'S', 'Title': 'Mr', 'FamilySize': 1, 'IsAlone': 1
    }
])

sample_preds = best_rf.predict(sample_passengers)
sample_probs = best_rf.predict_proba(sample_passengers)[:, 1]

for idx, (p, prob) in enumerate(zip(sample_preds, sample_probs)):
    res = 'SURVIVED (1)' if p == 1 else 'DIED (0)'
    print(f'Passenger {idx+1}: Prediction = {res} | Probability = {prob*100:.2f}%')

## 13. Kaggle Submission Generation
Generating the final `submission.csv` using the exact preprocessing and trained model.

In [ ]:
test_preds = best_rf.predict(X_test_kaggle)
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_preds
})
submission_df.to_csv('../submission/submission.csv', index=False)
print(f'Saved submission.csv with shape {submission_df.shape}')
submission_df.head(10)